<div class="alert alert-block alert-info">

# Part 3B: Machine Learning: A Better Way!
## Loading data for Training and Test Sets and Creating a Naïve Bayes Model using Pipelines

In Part 3A, you created training and test sets after removing MACCS Keys that had zero variance. This was useful because it showed how low-variance feature removal works. However, it also created a practical problem: once those MACCS Keys were removed, any new molecule used for prediction also had to be filtered in exactly the same way. In other words, we had to keep track of which MACCS Keys were removed and make sure those same features were removed from every new fingerprint before applying the model.

A better way to handle this is to save the model as part of a pipeline. A pipeline is a scikit-learn tool that bundles preprocessing steps and the trained model together into a single reusable object. This helps prevent common mistakes when applying a model to new data, such as forgetting to apply the same feature selection step that was used during training.

In this case, the pipeline will include two main steps: first, `VarianceThreshold` will remove MACCS Keys with zero variance; then, the classifier, such as `BernoulliNB`, will use the remaining MACCS Keys to make the prediction. Once the pipeline is trained, it can be saved using `joblib.dump()` and later reloaded with `joblib.load()`. This means that future predictions can be made in a new notebook without manually remembering which MACCS Keys were removed.

Importantly, the pipeline must be trained using the full original MACCS Key feature set, not a version where the zero-variance features have already been removed. If the training data already have only 163 MACCS Keys instead of the original 167, then the pipeline cannot learn which four original features should be removed. It only sees the already-filtered data.

Unfortunately, our current downsampled training and test sets were already created after removing the zero-variance MACCS Keys. Before we build the pipeline, we therefore need to recreate the training and downsampled training sets using the full MACCS Key feature list. Thankfully, we previously saved the activity data and MACCS Keys together in a CSV file. We can reload that file, rebuild the training and test sets from the full MACCS Key data, redo the downsampling, and then train the pipeline correctly.

### Loading the data into X and y.

Once again we need to load the saved activity/complete MACCS Keys into a dataframe.

In [ ]:
import pandas as pd
df_data = pd.read_csv("AID743139_activity_MACCS.csv")

In [ ]:
# we now have a dataframe with CIDS, activities and all 167 maccs keys
df_data.head(3)

We put the MACCS Keys into a variable called X_MACCS.<br>
We put the activity values into a variable called y.

In [ ]:
X_MACCS = df_data.iloc[:,2:] # this is dropping cid and activity and creating a new variable for maccs data
y = df_data['activity'].values

In [ ]:
X_MACCS.head(3)

### Train-Test-Split (a 9:1 ratio)

Now that we’ve reloaded the dataset, the next step is to divide it into two parts: one for training the model and one for testing it. 

Once again we will split the data so that 90% goes into the training set and 10% into the test set. The training set is used to build the model, while the test set is used to evaluate how well the model generalizes to new data. The difference here is that we are leaving in all the features with zero variance as we will remove those later when creating the pipeline.

Once again, `X_train` and `X_test` contain the molecular fingerprint data used by the model. The X values are the input features, meaning the structural information for each molecule. `X_train` contains the compounds used to train the model, while `X_test` contains compounds held back to evaluate how well the model performs on data it has not seen before. The variables `y_train` and `y_test` contain the known activity labels for those same compounds. `y_train` provides the activity labels used during training, and `y_test` provides the correct answers used to evaluate the model’s predictions. Since `test_size=0.1`, 10% of the dataset is placed in the test set and the remaining 90% is used for training.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = \
    train_test_split(X_MACCS, y, shuffle=True, random_state=3100, stratify=y, test_size=0.1) # test_size = 0.1 is 10% of the data set

print("Training set shape:", X_train.shape, y_train.shape)
print("where there are", X_train.shape[0], "samples, and", X_train.shape[1], "features")
print("and", y_train.shape[0], "activities associated with the training set.")
print()
print("Test set shape:", X_test.shape, y_test.shape)
print("where there are", X_test.shape[0], "samples, and", X_test.shape[1], "features")
print("and", y_test.shape[0], "activities associated with the test set.")
print()
print("Number of active compounds in training set:", y_train.sum())
print("Number of active compounds in test set:", y_test.sum())
print()
print("# inactives in training set: ", len(y_train) - y_train.sum())
print("# actives in training set:   ", y_train.sum())
ratio = (len(y_train) - y_train.sum())/y_train.sum()
print("the ratio of inactive to active in training set=", ratio)

### Balancing the training set

Recall from before that we have an unbalanced training set. We will create a balanced training set to ensure the model can learn to distinguish between both classes (active vs inactive) effectively.

In [ ]:
# load the numpy libraray
import numpy as np

# Indicies of each class' observations
idx_inactives = np.where( y_train == 0 )[0]
idx_actives   = np.where( y_train == 1 )[0]

# Number of observations in each class
num_inactives = len(idx_inactives)
num_actives   = len(idx_actives)

# Randomly sample from inactives without replacement
# setting size to the number of actives ensures we downsample inactives to match the number of actives
np.random.seed(0)  #sets the random seed for reproducibility. You might want to try a value of 2026 after you get your first confusion matix.
idx_inactives_downsampled = np.random.choice(idx_inactives, size=num_actives, replace=False)

# Join together downsampled inactives with actives
# vstack and hstack are used to combine arrays vertically and horizontally, respectively
# this ensures that the rows from downsampled inactives and actives are combined correctly
# we use vstack for a 2D array (X_train) and hstack for a 1D array (y_train)
X_train_downsampled = np.vstack((X_train.iloc[idx_inactives_downsampled], X_train.iloc[idx_actives])) #iloc resolved panda vs numpy issue
y_train_downsampled = np.hstack((y_train[idx_inactives_downsampled], y_train[idx_actives]))

#confirm the downsampling worked
print("# inactives orig: ", len(y_train) - y_train.sum())
print("# inactives downsampled: ", len(y_train_downsampled) - y_train_downsampled.sum())

print("# actives orig  : ", y_train.sum())
print("# actives downsampled  : ", y_train_downsampled.sum())
ratio = (len(y_train) - y_train.sum())/y_train.sum()
print("the ratio of active to inactive original=", ratio)
ratio_downsampled = (len(y_train_downsampled) - y_train_downsampled.sum())/y_train_downsampled.sum()
print("the ratio of active to inactive downsampled=", ratio_downsampled)
print()
# check to see the number of samples and features in the training set
print("Training set shape:", X_train.shape, y_train.shape)
print("where there are", X_train.shape[0], "samples, and", X_train.shape[1], "features")
print("and", y_train.shape[0], "activities associated with the training set.")
print()
print("Downsampled Training set shape:", X_train_downsampled.shape, y_train_downsampled.shape)
print("where there are", X_train_downsampled.shape[0], "samples, and", X_train_downsampled.shape[1], "features")
print("and", y_train_downsampled.shape[0], "activities associated with the training set.")

We are now ready to build the **Pipeline** that takes care of our zero variance issue with new molecules.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.naive_bayes import BernoulliNB
import joblib

pipe = Pipeline([
    ("var", VarianceThreshold(threshold=0.0)),
    ("clf", BernoulliNB())
])

pipe.fit(X_train_downsampled, y_train_downsampled)
joblib.dump(pipe, "nb_ds_maccs167_pipeline.joblib")

We can now use this pipeline to predict the activity of a molecule as well as a probability prediction.

The probability prediction gives information about the model’s confidence in its classification. If the model assigns a high probability to the correct class, this suggests that the prediction is reliable for that compound. If the model assigns probabilities close to 0.50, the model is uncertain, even if the final prediction is correct. If the model assigns a high probability to the wrong class, this is a warning sign that the model is confidently misclassifying the compound and may not be reliable for molecules like that one.

Note: The following code cell will return a warning that indicates the model was trained with named columns, but the new prediction input does not have those names. The warning is usually harmless.

In [ ]:
# load necessary rdkit libraries
from rdkit import Chem
from rdkit.Chem import MACCSkeys
from rdkit.DataStructs import ConvertToNumpyArray

In [ ]:
pipe = joblib.load("nb_ds_maccs167_pipeline.joblib")
print("Pipeline loaded successfully!")
print()
# Create a fingerprint for a molecule
smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"  # aspirin
mol = Chem.MolFromSmiles(smiles)
fp = MACCSkeys.GenMACCSKeys(mol)

# prepare the array based on the fingerprint
X_query = np.array(fp).reshape(1, -1) # The classifier needs a 2D shape, but we have 1D list. We reshape (1, n_bits)
pred = pipe.predict(X_query)  # no manual masking needed
if pred == 0:
    print("Molecule is inactive for human aromatase.")
else:
    print("Molecule is active for human aromatase. Active may be agonist or antagonist in this model.")

# Returns the model's confidence for each possible class (e.g., [P(Inactive), P(Active)]).
probability = pipe.predict_proba(X_query)

print("Probabilities (Inactive, Active):", probability)

In [ ]:
# We can also loop through a list to predict multiple activities.

#new_smiles = "C1=CC=C(C(=C1)C2=NC(=NO2)C3=CC=NC=C3)Cl" #CID 65758 should be active
#new_smiles = "C1=CC(=CC=C1C2=COC3=CC(=CC(=C3C2=O)O)O)O" #CID 5280961 should be active
#new_smiles = "CN(C1CCN(CC1)C2=NC3=CC=CC=C3N2CC4=CC=C(C=C4)F)C5=NC=CC(=O)N5" #CID 65906 should be INactive
#new_smiles = "C1=CNC(=O)NC1=O" #CID 1174 should be INactive
#new_smiles = "C[C@H]1C[C@@H](C(=O)[C@@H](C1)[C@@H](CC2CC(=O)NC(=O)C2)O)C" #CID 6197 should be active

smiles_list = ["C1=CC=C(C(=C1)C2=NC(=NO2)C3=CC=NC=C3)Cl", "C1=CC(=CC=C1C2=COC3=CC(=CC(=C3C2=O)O)O)O", 
               "CN(C1CCN(CC1)C2=NC3=CC=CC=C3N2CC4=CC=C(C=C4)F)C5=NC=CC(=O)N5","C1=CNC(=O)NC1=O", 
               "C[C@H]1C[C@@H](C(=O)[C@@H](C1)[C@@H](CC2CC(=O)NC(=O)C2)O)C"]
fps = []
for s in smiles_list:
    mol = Chem.MolFromSmiles(s)
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=int)
    ConvertToNumpyArray(fp, arr)
    fps.append(arr)

X_batch = np.array(fps)
preds = pipe.predict(X_batch)
print(preds)
preds = pipe.predict_proba(X_batch)
print(preds)

Notice that our predictions no longer have the Warning about Variance Threshold names! So this is a much more efficient and effective way to generate a model.

Finally, let's store the downsampled complete 167 MACCS Keys datasets for later use.

In [ ]:
np.save("X_train_downsampled_MACCS167.npy", X_train_downsampled)
np.save("y_train_downsampled_MACCS167.npy", y_train_downsampled)
np.save("X_test_MACCS167.npy", X_test)
np.save("y_test_MACCS167.npy", y_test) 

Since this is a much clearner way to code your ML model. Use this method for `08_3_ML_NB_homework.ipynb`

Let's make sure our model behave the same way with the pipeline.

In [ ]:
from sklearn.metrics import classification_report #provides detailed report that includes precision and sensitivity
from sklearn.metrics import confusion_matrix      # gives a 2x2 matrix for true netatives, false positives, false negatives and true positives
from sklearn.metrics import accuracy_score        # computes accuracy = number of correct preictions/total number of preditions
from sklearn.metrics import roc_auc_score         # Computs Area under the ROC curve, evaluates trade-off of true positive rate and false positive rate

In [ ]:
X_eval = X_test.to_numpy() # convert to numpy to avoid warning with no column names
y_true = y_test
set_name = "Test set"

y_pred = pipe.predict(X_eval)




In [ ]:
CMat = confusion_matrix( y_true, y_pred )    #-- generate confusion matrix
print(CMat)    # [[TN, FP], 
               #  [FN, TP]]

# Extracting TN, FP, FN, TP from the confusion matrix               
TN = CMat[0, 0]  # True Negatives
FP = CMat[0, 1]  # False Positives
FN = CMat[1, 0]  # False Negatives
TP = CMat[1, 1]  # True Positives

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

print("Total predictions:", TN + FP + FN + TP)

In [ ]:
y_score = pipe.predict_proba(X_eval)[:, 1]

TN, FP, FN, TP = confusion_matrix(y_true, y_pred).ravel()

acc  = accuracy_score(y_true, y_pred)
prec = TP / (TP + FP)
sens = TP / (TP + FN)
spec = TN / (TN + FP)
bacc = (sens + spec) / 2
f1_score = 2 * (prec * sens) / (prec + sens)

auc = roc_auc_score(y_true, y_score)

print(f"{set_name} performance metrics:")
print(f"Accuracy          = {acc:.4f}")
print(f"Precision         = {prec:.4f}")
print(f"Sensitivity       = {sens:.4f}")
print(f"Specificity       = {spec:.4f}")
print(f"Balanced Accuracy = {bacc:.4f}")
print(f"F1 Score          = {f1_score:.4f}")
print(f"AUC-ROC           = {auc:.4f}")


<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong> Write your answer in the following raw cell.
    
How do these values compare to the test set values in 8_3A_ML_NB_activity.ipynb? Does using the pipeline with the full 167 features result in the same or different analysis? Explain.